# NimbusPay ticket triage fine-tuning

Base model: `Qwen/Qwen2.5-1.5B-Instruct` (1.5B parameters, system-role chat template). Training uses PEFT LoRA/QLoRA; the 4-bit version is the primary run so it fits a free Colab T4. Dev is used only for evaluation/model selection; test is used only for final inference.

AI assistant use: Copilot SDK in VS Code helped draft the audit and notebook code. It did not generate training labels, dev labels, or test predictions. All reported measurements must come from executing this notebook.

## 0. Colab setup

Copy `audit_clean.py`, `score.py`, and the provided `data/` directory to the Drive folder below before running. The raw training data and base model are not part of the submission repository.

In [ ]:
%pip install -q 'transformers>=4.45,<5' peft bitsandbytes accelerate

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
WORK_DIR = Path('/content/drive/MyDrive/nimbuspay-assignment')
DATA_DIR = WORK_DIR / 'data'
OUT_DIR = WORK_DIR / 'submission'
RESULTS_DIR = OUT_DIR / 'experiments'
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert (DATA_DIR / 'train.jsonl').exists()
assert (DATA_DIR / 'dev.jsonl').exists()
assert (DATA_DIR / 'test_inputs.jsonl').exists()


## 1. Data audit and cleaning

`audit_clean.py` contains the data-only repair rules. It drops unparseable/truncated assistant labels, normalizes recoverable transaction identifiers and dates, fixes schema/rule-inconsistent priority and human-escalation labels, and records each changed row. The dev and test files are never passed to the cleaner.

In [ ]:
import csv, json, sys
from collections import Counter
sys.path.insert(0, str(WORK_DIR))
from audit_clean import clean_rows
import score as official_score

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_raw = load_jsonl(DATA_DIR / 'train.jsonl')
dev = load_jsonl(DATA_DIR / 'dev.jsonl')
test = load_jsonl(DATA_DIR / 'test_inputs.jsonl')
train_clean, cleaning_log, audit_stats = clean_rows(train_raw)
with open(OUT_DIR / 'cleaning_log.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'action', 'reason'])
    writer.writerows(cleaning_log)
print(audit_stats)
print('actions:', dict(Counter(action for _, action, _ in cleaning_log)))
print('train/dev/test:', len(train_raw), len(dev), len(test))


## 2. Tokenizer, length evidence, and scoring

The context length is derived from the longest cleaned training example after chat templating, rounded to a multiple of 128 and capped at 1,536. This avoids truncating any training target while retaining a T4-safe bound; review the printed distribution and the assertion before training.

In [ ]:
import math, time, gc, torch
from transformers import AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def chat_messages(row):
    return [{'role': msg['role'], 'content': msg['content']} for msg in row['messages'] if msg['role'] in ('system', 'user', 'assistant')]

lengths = [len(tokenizer.apply_chat_template(chat_messages(row), tokenize=True, add_generation_prompt=False)) for row in train_clean]
lengths.sort()
print({'n': len(lengths), 'p50': lengths[len(lengths)//2], 'p95': lengths[int(.95*(len(lengths)-1))], 'p99': lengths[int(.99*(len(lengths)-1))], 'max': max(lengths)})
MAX_LENGTH = min(1536, max(256, math.ceil(max(lengths) / 128) * 128))
assert max(lengths) <= MAX_LENGTH, 'Increase MAX_LENGTH; do not silently truncate answer labels.'
print('MAX_LENGTH:', MAX_LENGTH)

KEYS = ['category', 'priority', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human']
def score_predictions(gold_rows, pred_rows):
    pred_by_id = {p['id']: p.get('output') for p in pred_rows}
    field_hits = Counter()
    category_counts, category_exact = Counter(), Counter()
    exact, valid = 0, 0
    failures = []
    for row in gold_rows:
        gold = json.loads(row['messages'][-1]['content'])
        raw = pred_by_id.get(row['id'])
        try:
            pred = json.loads(raw.strip())
            is_valid = isinstance(pred, dict)
        except (json.JSONDecodeError, AttributeError):
            pred, is_valid = None, False
        valid += int(is_valid)
        hits = {key: is_valid and key in pred and type(pred[key]) is type(gold[key]) and pred[key] == gold[key] for key in KEYS}
        is_exact = is_valid and all(hits.values()) and set(pred) == set(KEYS)
        exact += int(is_exact)
        category = gold['category']
        category_counts[category] += 1
        category_exact[category] += int(is_exact)
        field_hits.update({key: int(hit) for key, hit in hits.items()})
        if not is_exact:
            failures.append({'id': row['id'], 'category': category, 'ticket': row['messages'][1]['content'], 'gold': gold, 'prediction': pred, 'wrong_fields': [key for key, hit in hits.items() if not hit]})
    n = len(gold_rows)
    official = official_score.score(gold_rows, pred_rows)['all']
    assert official['exact_match'] == exact / n
    assert official['mean_field_acc'] == sum(field_hits.values()) / (n * len(KEYS))
    return {'exact_match': official['exact_match'], 'mean_field_accuracy': official['mean_field_acc'], 'json_valid': official['json_valid'], 'fields': official['fields'], 'categories': {key: {'n': category_counts[key], 'exact_match': category_exact[key] / category_counts[key]} for key in category_counts}, 'failures': failures}


## 3. Greedy baselines

Both baselines use the same 4-bit base model and greedy `generate`. The fixed-prompt baseline uses the data's system/user messages; the reference baseline adds the schema rules to the system message. The latter is not used for training or final inference.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def load_base(quantized=True):
    kwargs = {'device_map': 'auto', 'torch_dtype': torch.float16}
    if quantized:
        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)

SCHEMA_RULES = '''Return exactly one JSON object with category, priority, amount_paise, txn_date, txn_id, channel, language, needs_human. Categories: payment_failed, refund, fraud, account_access, kyc, offers, other. Fraud always P1. P1 also applies to amount >= 50000 rupees or legal threats mentioning RBI, ombudsman, consumer court, lawyer, legal notice, or legal action. Otherwise P2 applies to payment_failed/refund of at least 5000 rupees or account_access; else P3. needs_human is true for P1 or explicit prior contact about this issue. Transaction fields are null for account_access, kyc, and other. Convert amounts to paise, resolve dates against Received, normalize NimbusPay IDs as NP plus 10 digits, and use null when absent. channel is only explicit upi, card, netbanking, or wallet. language is hi-en for Hinglish, en otherwise.'''

def generate_one(model, row, add_rules=False):
    msgs = chat_messages(row)[:2]
    if add_rules:
        msgs[0] = dict(msgs[0], content=msgs[0]['content'] + '\n' + SCHEMA_RULES)
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, inputs.shape[-1]:], skip_special_tokens=True).strip()

def predict(model, rows, add_rules=False):
    return [{'id': row['id'], 'output': generate_one(model, row, add_rules)} for row in rows]

base = load_base(quantized=True)
base_fixed = predict(base, dev, add_rules=False)
base_rules = predict(base, dev, add_rules=True)
print('fixed prompt:', {k:v for k,v in score_predictions(dev, base_fixed).items() if k != 'failures'})
print('rules prompt:', {k:v for k,v in score_predictions(dev, base_rules).items() if k != 'failures'})
del base, base_fixed, base_rules
gc.collect(); torch.cuda.empty_cache()


## 4. Answer-only LoRA/QLoRA training and controlled experiments

The prefix (system + user + assistant-generation header) is masked with `-100`; loss is computed only on the assistant response. The primary configuration is cleaned data, 4-bit NF4, rank 8, and two epochs. Each experiment below changes one factor relative to that configuration. The additional lower-learning-rate run is the own-idea experiment, motivated by reducing overfitting. Peak memory and wall-clock training time are measured during each run.

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.items = []
        for row in rows:
            messages = chat_messages(row)
            full = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
            prefix = tokenizer.apply_chat_template(messages[:2], tokenize=True, add_generation_prompt=True)
            if full[:len(prefix)] != prefix:
                raise ValueError(f'Chat-template prefix mismatch for {row["id"]}; refusing to train on misaligned labels.')
            if len(full) > MAX_LENGTH:
                raise ValueError(f'Training row {row["id"]} exceeds evidence-based MAX_LENGTH.')
            labels = list(full)
            labels[:len(prefix)] = [-100] * len(prefix)
            self.items.append({'input_ids': full, 'labels': labels})
    def __len__(self): return len(self.items)
    def __getitem__(self, index): return self.items[index]

def collate(batch):
    width = max(len(item['input_ids']) for item in batch)
    ids = torch.full((len(batch), width), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((len(batch), width), -100, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, item in enumerate(batch):
        size = len(item['input_ids'])
        ids[i, :size] = torch.tensor(item['input_ids'])
        labels[i, :size] = torch.tensor(item['labels'])
        mask[i, :size] = 1
    return {'input_ids': ids, 'attention_mask': mask, 'labels': labels}

def run_experiment(name, rows, quantized=True, rank=8, epochs=2, learning_rate=2e-4):
    torch.manual_seed(42)
    model = load_base(quantized=quantized)
    if quantized:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    config = LoraConfig(r=rank, lora_alpha=2*rank, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model = get_peft_model(model, config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert trainable <= 50_000_000, f'{trainable:,} trainable parameters exceeds assignment limit'
    model.print_trainable_parameters()
    model.config.use_cache = False
    torch.cuda.reset_peak_memory_stats()
    args = TrainingArguments(output_dir=str(RESULTS_DIR / name / 'trainer'), per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=epochs, learning_rate=learning_rate, fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False}, optim='paged_adamw_8bit' if quantized else 'adamw_torch', logging_steps=25, save_strategy='no', report_to='none', seed=42, data_seed=42, remove_unused_columns=False)
    trainer = Trainer(model=model, args=args, train_dataset=TicketDataset(rows), data_collator=collate)
    start = time.perf_counter()
    trainer.train()
    minutes = (time.perf_counter() - start) / 60
    peak_gib = torch.cuda.max_memory_reserved() / (1024**3)
    adapter_dir = RESULTS_DIR / name / 'adapter'
    adapter_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(adapter_dir, safe_serialization=True)
    trainer.model.eval()
    torch.cuda.empty_cache()
    predictions = predict(trainer.model, dev)
    metrics = score_predictions(dev, predictions)
    summary = {'experiment': name, 'quantized_4bit': quantized, 'rank': rank, 'epochs': epochs, 'learning_rate': learning_rate, 'trainable_params': trainable, 'exact_match': metrics['exact_match'], 'mean_field_accuracy': metrics['mean_field_accuracy'], 'peak_gpu_memory_gib': peak_gib, 'training_minutes': minutes}
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    return summary, metrics, predictions

experiments = [
    ('clean_qlora_r8_2ep', train_clean, True, 8, 2),
    ('raw_qlora_r8_2ep', train_raw, True, 8, 2),
    ('clean_fp16_lora_r8_2ep', train_clean, False, 8, 2),
    ('clean_qlora_r16_2ep', train_clean, True, 16, 2),
    ('clean_qlora_r8_1ep', train_clean, True, 8, 1),
]
results, dev_details, dev_predictions = [], {}, {}
for name, rows, quantized, rank, epochs in experiments:
    summary, metrics, predictions = run_experiment(name, rows, quantized, rank, epochs)
    results.append(summary)
    dev_details[name] = metrics
    dev_predictions[name] = predictions
    print(summary)

summary, metrics, predictions = run_experiment('clean_qlora_r8_lr1e4', train_clean, True, 8, 2, learning_rate=1e-4)
results.append(summary)
dev_details[summary['experiment']] = metrics
dev_predictions[summary['experiment']] = predictions
print(summary)

import pandas as pd
results_table = pd.DataFrame(results)
display(results_table)
results_table.to_csv(OUT_DIR / 'experiment_results.csv', index=False)


## 5. Select on dev, error analysis, and test predictions

After the experiments complete, choose the run with the highest dev exact match (use mean field accuracy as a tie-breaker). Review at least ten concrete dev failures and write a human explanation for each in the report; do not infer a cause solely from an aggregate score. The cells below generate the evidence table and save reproducible test predictions.

In [ ]:
best = max(results, key=lambda row: (row['exact_match'], row['mean_field_accuracy']))
best_name = best['experiment']
print('selected from dev:', best)
failures = dev_details[best_name]['failures']
print('weakest fields:', sorted(dev_details[best_name]['fields'].items(), key=lambda item: item[1]))
print('category exact match:', dev_details[best_name]['categories'])
for item in failures[:10]:
    print({'id': item['id'], 'category': item['category'], 'wrong_fields': item['wrong_fields'], 'ticket': item['ticket'], 'gold': item['gold'], 'prediction': item['prediction'], 'explanation_to_complete': ''})

best_adapter = RESULTS_DIR / best_name / 'adapter'
OUT_ADAPTER = OUT_DIR / 'adapter'
OUT_ADAPTER.mkdir(parents=True, exist_ok=True)
import shutil
for file in best_adapter.iterdir():
    shutil.copy2(file, OUT_ADAPTER / file.name)
(OUT_ADAPTER / 'base_model_id.txt').write_text(MODEL_ID, encoding='utf-8')

final_model = load_base(quantized=True)
final_model = PeftModel.from_pretrained(final_model, OUT_ADAPTER)
final_model.eval()
test_predictions = predict(final_model, test)
with open(OUT_DIR / 'predictions_test.jsonl', 'w', encoding='utf-8') as f:
    for item in test_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# Greedy decoding should be deterministic: verify a fixed sample against the saved outputs.
sample = test[:min(10, len(test))]
reproduced = predict(final_model, sample)
assert reproduced == test_predictions[:len(sample)]
print('saved', len(test_predictions), 'test predictions; reproducibility sample passed')


## Results and limitations

Do not submit placeholder or unexecuted metrics as results. The audit currently reports 1,445 raw rows, 1,420 retained rows, 25 dropped truncated/invalid labels, and 157 repaired rows (182 log entries total); those counts must be regenerated from the supplied data by the audit cell. Baseline, experiment, error-analysis, adapter, and test-prediction outputs remain pending until this notebook has run on a Colab T4.